# CrewAI × MLflow マルチエージェント可観測性サンプル

MLflowブログ「[Seeing Inside Your Multi-Agent System with MLflow](https://mlflow.org/blog/observability-multi-agent-part-1/)」のローカル例を、Databricksのマネージド MLflow と基盤モデルで動くように書き換えたものです。

# このノートブックの背景

マルチエージェントを組むこと自体は難しくありません。難しいのは、エージェント同士が会話を始めて初めて表面化する subtle な失敗を捕まえることです。たとえば、あるエージェントの誤った数値が下流に伝播するカスケードエラー、ハルシネーションが共有コンテキストに書き込まれて「事実」になる共有メモリ汚染、応答待ちのまま静かに停止するループなどです。

単一エージェントなら入出力を見れば足りますが、マルチエージェントでは分析の単位が「プロンプト」から「状態遷移 (state transition)」へ移ります。これが元記事の言う「モニタリングから可観測性へ」の転換です。

そのため本ノートブックは二層構成を取ります。トレースが「何が起きたか」を、メトリクスが「それが許容範囲だったか」を教えます。

1. autolog による自動トレースで、オーケストレーターからワーカーへのネストされた実行グラフを可視化する (土台)
2. その上に3つのメトリクスの柱 (ルーティング・状態ハンドオフ・運用テレメトリ) をカスタム属性として重ね、挙動の良し悪しを判定する

なお元記事はシリーズのPart 1で、ここで扱うのは受動的な観察までです。安全性の強制やコスト管理といった能動的なステアリング (ガバナンス) はPart 2の範囲で、本ノートブックには含めていません。

# 実装の進め方 (ロードマップ)

注意点として、上の「3つの柱」は測りたいものの分類であり、以降のセルの並びはコードの実行順 (依存関係) です。この2つの軸は一致しません。たとえば柱2を測るには受け渡し内容を保持する入れ物が先に要る、といった都合で、柱の名前と見出しがずれて見えることがあります。迷子にならないよう、対応を先に示します。

1. 共有状態とハンドオフ効率スコア: 柱2(状態一貫性)を測るための道具を用意します。柱そのものではなく、後続セルが依存する部品を先に定義する準備段階です。
2. ルーティング追跡と状態ハンドオフ: 1で用意した道具を使い、柱1(ルーティング)と柱2の記録ロジックを定義します。どちらもコールバックで実装するためまとめています。
3. エージェントとタスクの定義: 2の記録ロジックを実際のエージェントとタスクに結びつけ、コールバックを挿し込みます。
4. 運用テレメトリでクルー全体をラップ: 柱3です。個々のステップではなく、クルーの実行全体を1スパンで包んでサマリ指標を取ります。
5. 異常系: わざと失敗を注入し、これらのメトリクスが点灯することを確認します。

参考: [CrewAIトレース (Databricks 日本語ドキュメント)](https://docs.databricks.com/gcp/ja/mlflow3/genai/tracing/integrations/crewai)

In [0]:
%pip install --upgrade "mlflow[databricks]>=3.1" crewai nest_asyncio
%restart_python

INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 182.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 133.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 137.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 157.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

# セットアップ

トラッキングサーバーを自前で立てる必要はありません。トラッキングURIを `databricks` にして、自動トレースを有効化します。LLMはDatabricksのOpenAI互換エンドポイント (基盤モデル) を CrewAI から LiteLLM 経由で呼び出します。

ここで `mlflow.crewai.autolog()` が、CrewAIワークフローのネストされたトレース (各タスクと実行エージェント、すべてのLLM呼び出しの入出力、各操作のレイテンシー) を自動的にキャプチャします。これがブラックボックスになりがちなマルチエージェントの実行を、ナビゲートできる状態遷移のマップに変える土台です。サーバーレスコンピュートでは自動有効化されないため、この一行を明示的に呼ぶ必要があります。

In [0]:
import os
import time
import re
import mlflow
from crewai import Agent, Crew, Task, LLM, Process

# Databricksノートブックのカーネルは既にイベントループを動かしているため、
# 同期 kickoff() を許可するためにループのネストを有効化する
# (async に切り替えるとMLflowのCrewAIトレースが取得できないため、この方法を使う)
import nest_asyncio
nest_asyncio.apply()

# CrewAIワークフローのネストされたトレースを自動キャプチャ
mlflow.crewai.autolog()

# DatabricksのマネージドMLflowにトレースを記録
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/crewai-observability-demo")

# ノートブックコンテキストからホストとトークンを取得
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

# CrewAI の既知の不具合 (issue #5886) 対応:
# 非Anthropicプロバイダー (OpenAI互換 / Databricks基盤モデル等) に対しても
# cache_breakpoint がメッセージに差し込まれ、"unknown field" として弾かれる。
# breakpoint を付与する関数を無効化して回避する。将来のバージョンで修正されたら不要。
try:
    import crewai.llms.cache as _crewai_cache
    _crewai_cache.mark_cache_breakpoint = lambda msg: msg
except Exception as _e:
    print(f"cache_breakpoint パッチをスキップしました: {_e}")

# DatabricksのOpenAI互換エンドポイント (/serving-endpoints) 経由でLLMを呼ぶ。
# litellmの databricks/ プロバイダーは一部バージョンでプロンプトキャッシュ用の
# フィールド (cache_breakpoint 等) をリクエストに差し込み、基盤モデルの
# サービングエンドポイントに "unknown field" として弾かれることがある。
# openai/ プロバイダーで素のchat completionsとして呼ぶとこの変換を回避できる。
# 本番ではトークンをハードコードせず、Databricksシークレットまたは AI Gateway を使ってください。
llm = LLM(
    model="openai/databricks-llama-4-maverick",  # 環境に合わせてエンドポイント名を変更
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
    api_key=DATABRICKS_TOKEN,
    temperature=0.2,
)

/databricks/python_shell/lib/dbruntime/autoreload/discoverability/autoreload_discoverability_hook.py:72: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `Settings` to V2.
  return orig_warn(*args, **kwargs)
2026/06/16 03:45:19 ERROR mlflow.crewai: An exception happens when applying auto-tracing to crewai. Exception: No module named 'crewai.agents.agent_builder.base_agent_executor_mixin'
If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


# 共有状態とハンドオフ効率スコア

ロードマップの1番目です。概要で挙げた3つの柱のうち、まず柱2(状態一貫性)を測るための土台をここで用意します。これは柱そのものではなく、次のセル以降が依存する道具(エージェント間の受け渡しを保持する共有状態と、引き継ぎ度合いを測るスコア関数)を先に定義する準備段階です。

マルチエージェントでは分析の単位が状態遷移に移るため、エージェント間で「何が受け渡されたか」を見えるようにすることが要になります。ここはカスケードエラーや共有メモリ汚染が顕在化する箇所です。共有メモリ汚染とは、あるエージェントのハルシネーションが後続エージェントにとっての「事実」になり、品質が突然ではなく徐々に劣化していく現象です。どこで始まったか特定しづらいため、早めにシグナルを取ることが効きます。

そこで、エージェント間で受け渡される中間結果を保持する簡易ステートと、追加のLLM呼び出しなしで計算できる引き継ぎ度合いのスコアを定義します。スコアは前段の出力が次段の入力にどれだけ含まれるかを文字bigramの重複で測り、1.0に近いほどそのまま引き継がれ、0.0に近いほど取りこぼされていることを意味します。日本語は空白で単語区切りされないため、元記事の空白区切りトークンではなく文字bigramを使っています。

In [0]:
class CrewState:
    """エージェント間で共有する中間結果のシンプルなストア。"""

    def __init__(self) -> None:
        self._state: dict[str, str] = {}

    def set_state(self, key: str, value: str) -> None:
        self._state[key] = value

    def get_state(self, key: str) -> str | None:
        return self._state.get(key)

    @staticmethod
    def task_output_to_text(output: object) -> str:
        # CrewAIのTaskOutputは raw に本文を持つ
        return getattr(output, "raw", str(output))


crew_state = CrewState()


def _char_bigrams(text: str) -> set[str]:
    # 空白・改行を除いた文字列から2-gramの集合を作る。日本語は空白で単語区切り
    # されないため、単語分割ではなく文字bigramで重複を測る (英語にもそのまま適用可)。
    s = "".join(text.lower().split())
    if len(s) < 2:
        return {s} if s else set()
    return {s[i : i + 2] for i in range(len(s) - 1)}


def handoff_efficiency_score(output_text: str, next_input_text: str) -> float:
    """前段の出力が次段の入力にどれだけ引き継がれたかを文字bigramの重複で測る。
    1.0に近いほど前段の内容がそのまま次段に渡り、0.0に近いほど取りこぼされている。"""
    output_grams = _char_bigrams(output_text)
    input_grams = _char_bigrams(next_input_text)
    if not output_grams:
        return 0.0
    return len(output_grams & input_grams) / len(output_grams)


def extract_numeric_tokens(text: str) -> list[str]:
    """レポートとソースの数値突き合わせ (カスケードエラー検知) 用に数値を抽出。"""
    return re.findall(r"\d[\d,\.]*", text or "")

# メトリクスの柱1・2: ルーティング追跡と状態ハンドオフ

ロードマップの2番目です。1つ前のセルで用意した道具を使い、柱1(ルーティング)と柱2(状態一貫性)の記録ロジックを定義します。両者は「エージェントのステップやタスクの切れ目にコールバックを差し込む」という共通の仕組みで実装できるため、1つのセルにまとめています。実際にエージェントへ結びつけるのは次のセルです。

柱1のルーティングは「スーパーバイザーは適切な相手に、効率よくタスクを振っているか」を見ます。`step_callback` は各エージェントステップの後に発火するので、ここで委任回数やループの兆候を記録します。委任が正しかったか、冗長ではないか、無駄なループに入っていないか、が観点です。

柱2の状態一貫性は、マルチエージェント最大のリスク、つまり「あるエージェントが微妙に誤った出力を出し、それが下流で真実として扱われる」状況を捉えます。`log_state_handoff` は `@mlflow.trace` で独立したスパンを作り、前段から次段への引き継ぎ効率を属性として残します。コンテキストを渡しすぎても渡さなすぎてもシグナルになります。

In [0]:
delegation_counts: dict[str, int] = {}


@mlflow.trace(name="state_handoff", span_type="CHAIN")
def log_state_handoff(from_agent: str, to_agent: str, output: str, next_input: str) -> None:
    span = mlflow.get_current_active_span()
    if span:
        span.set_attributes({
            "handoff.from_agent": from_agent,
            "handoff.to_agent": to_agent,
            "handoff.output_chars": len(output),
            "handoff.input_chars": len(next_input),
            "handoff.efficiency_score": handoff_efficiency_score(output, next_input),
        })


def track_routing(step_output) -> None:
    agent_name = getattr(step_output, "agent", "unknown")
    delegation_counts[agent_name] = delegation_counts.get(agent_name, 0) + 1
    span = mlflow.get_current_active_span()
    if span:
        span.set_attributes({
            f"routing.{agent_name}.call_count": delegation_counts[agent_name],
            "routing.total_delegations": sum(delegation_counts.values()),
            # 通常の推論 + ツール + リトライで3回程度は許容し、それを超えたらループ疑い
            "routing.loop_detected": any(v > 3 for v in delegation_counts.values()),
        })

# エージェントとタスクの定義

ロードマップの3番目です。ここまでに定義した道具と記録ロジックを、実際のエージェントとタスクに結びつけます。各タスクの `callback` に柱1・2の記録関数を挿し込むことで、実行時に自動でメトリクスが記録される配線を作ります。

オーケストレーション役のプランナー、分析役、統合役の3エージェントによるリサーチパイプラインです。元記事と同じく、オーケストレーターがワーカーに委任し、結果が段から段へ受け渡される構造になっています。MLflow Tracing は、このオーケストレーターとワーカーの親子関係をネストされたスパンとして再構成します。各タスクの `callback` で出力を共有ステートに書き込み、前段から次段へのハンドオフ効率を記録します。

In [0]:
planner = Agent(
    role="リサーチプランナー",
    goal="与えられたテーマを分析可能な小問に分解し、調査計画を作る",
    backstory="複雑なテーマを構造化することに長けたオーケストレーター。",
    llm=llm,
    verbose=True,
)

analyst = Agent(
    role="アナリスト",
    goal="調査計画に沿って各小問へ根拠つきの分析を与える",
    backstory="定量・定性の両面から論点を掘り下げる専門家。",
    llm=llm,
    verbose=True,
)

synthesist = Agent(
    role="統合担当",
    goal="分析結果を1本の簡潔なレポートにまとめる",
    backstory="要点を落とさず読みやすく統合するライター。",
    llm=llm,
    verbose=True,
)

TOPIC = "国内製造業における生成AI導入の利点と課題"


def _on_plan_complete(output) -> None:
    text = crew_state.task_output_to_text(output)
    crew_state.set_state("plan", text)


def _on_analysis_complete(output) -> None:
    text = crew_state.task_output_to_text(output)
    crew_state.set_state("analysis", text)
    # プランナー出力がどれだけ分析に引き継がれたか
    plan = crew_state.get_state("plan") or ""
    log_state_handoff(
        from_agent="リサーチプランナー",
        to_agent="アナリスト",
        output=plan,
        next_input=text,
    )


def _on_synthesis_complete(output) -> None:
    text = crew_state.task_output_to_text(output)
    crew_state.set_state("synthesis", text)
    analysis = crew_state.get_state("analysis") or ""
    log_state_handoff(
        from_agent="アナリスト",
        to_agent="統合担当",
        output=analysis,
        next_input=text,
    )


plan_task = Task(
    description=f"テーマ「{TOPIC}」を3〜4個の調査小問に分解し、それぞれの狙いを述べてください。",
    expected_output="番号つきの調査小問リストと各狙い。",
    agent=planner,
    callback=_on_plan_complete,
)

analysis_task = Task(
    description="調査計画の各小問について、根拠とともに分析を述べてください。数値に言及する場合は明示してください。",
    expected_output="小問ごとの分析。",
    agent=analyst,
    context=[plan_task],
    callback=_on_analysis_complete,
)

synthesis_task = Task(
    description="分析結果を、利点と課題が一目で分かる簡潔な日本語レポートにまとめてください。",
    expected_output="800字程度のレポート。",
    agent=synthesist,
    context=[analysis_task],
    callback=_on_synthesis_complete,
)

# メトリクスの柱3: 運用テレメトリでクルー全体をラップ

ロードマップの4番目です。柱1・2がエージェントの各ステップを個別に記録するのに対し、柱3はクルーの実行全体を1つのスパンで包み、サマリ指標を取ります。

柱3が捉えるのは、「正しい答えが、壊滅的な運用上の失敗を覆い隠している」状況です。システムは正解を返しても、その裏で何度も再帰呼び出しし、リトライを重ね、応答待ちで時間を浪費しているかもしれません。トレースは何が起きたかを、メトリクスはそれが許容範囲だったかを教えます。成功した応答が高コストな実行を隠していないか、をここで見ます。

そこで `crew.kickoff()` をトレース付きスパンで包み、総実行時間・タスク数・各エージェントの出力量・数値突き合わせなどのサマリ指標を記録します。ループの誤検知を防ぐため、実行ごとに委任カウントをリセットします。本番では、ノードごとのトークンコスト、レイテンシーのボトルネック、タスク深度なども併せて取ると、リソースの浪費や無限ループを早期に発見できます。

In [0]:
import contextvars
from concurrent.futures import ThreadPoolExecutor


def _kickoff_sync_in_worker_thread(crew: Crew):
    """ノートブックの実行中イベントループを避けるため、ループを持たない
    ワーカースレッドで同期 kickoff() を実行する。
    現在のコンテキスト (アクティブなMLflowスパンを含む) をコピーして渡すことで、
    autologが作る Crew.kickoff スパンとの親子関係を維持する。"""
    ctx = contextvars.copy_context()
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(ctx.run, crew.kickoff)
        return future.result()


@mlflow.trace(name="crew_run_with_metrics", span_type="CHAIN")
def run_crew_with_metrics(crew: Crew):
    """crew.kickoff() をトレース付きスパン内で実行し、サマリ指標を収集する。"""
    delegation_counts.clear()  # 実行間でリセットしてループ誤検知を防ぐ
    t0 = time.perf_counter()
    result = _kickoff_sync_in_worker_thread(crew)
    duration_s = round(time.perf_counter() - t0, 3)

    span = mlflow.get_current_active_span()
    if span is not None:
        plan = crew_state.get_state("plan") or ""
        analysis = crew_state.get_state("analysis") or ""
        synthesis = crew_state.get_state("synthesis") or ""

        span.set_attributes({
            "crew.total_duration_s": duration_s,
            "crew.task_count": len(crew.tasks),
            "crew.agent_count": len(crew.agents),
            "planner.output_chars": len(plan),
            "analyst.output_chars": len(analysis),
            "synthesist.output_chars": len(synthesis),
            # レポートとソースの数値件数を突き合わせ、カスケードした数値の取りこぼしを検知
            "validation.report_number_count": len(extract_numeric_tokens(synthesis)),
            "validation.source_number_count": len(extract_numeric_tokens(analysis)),
        })
    return result


crew = Crew(
    agents=[planner, analyst, synthesist],
    tasks=[plan_task, analysis_task, synthesis_task],
    process=Process.sequential,
    step_callback=track_routing,  # 柱1: ルーティング追跡
    verbose=True,
)

result = run_crew_with_metrics(crew)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8e08660b-07d8-4de8-a0fd-ad6b7eb6b969                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  テーマ「国内製造業における生成AI導入の利点と課題」を3〜4個の調査小問に分解し、それぞれの狙いを述べてください   │
│  。                                                                                                             │
│  ID: 3f56a7b7-94d1-4d35-a694-b4fa7028468c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: リサーチプランナー                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│  テーマ「国内製造業における生成AI導入の利点と課題」を3〜4個の調査小問に分解し、それぞれの狙いを述べてください   │
│  。                                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: リサーチプランナー                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  テーマ「国内製造業における生成AI導入の利点と課題」を調査するために、以下の3つの調査小問に分解しました。        │
│                                                                                                                 │
│  1. **国内製造業における生成AIの導入状況はどのようなものか？**                                                  │
│     -                                                                                                           │
│  狙い：この小問では、国内製造業における生成AIの現在の導入状況を把握することを目的としています。導入の程度、業   │
│  界別の違い、導入している企業とそうでない企業の特徴などを明らかにすることで、生成AIが製造業に与える影響を理解   │
│  するための基礎データを収集します。                                                                             │
│                                                                                                                 │
│  2. **生成AIの導入が国内製造業にもたらす利点は何か？**                                                          │
│     -                                                                                                           │
│  狙い：この小問では、生成AIを導入することで国内製造業が享受できる具体的な利点を探ります。生産性の向上、コスト   │
│  削減、新製品開発の促進、品質管理の強化など、生成AIがもたらすと期待される利点を特定し、その実例や効果を分析し   │
│  ます。                                                                                                         │
│                                                                                                                 │
│  3. **国内製造業における生成AI導入の課題や制約は何か？**                                                        │
│     -                                                                                                           │
│  狙い：この小問では、生成AIの導入に伴う課題や制約を明らかにすることを目的としています。技術的な制約、コスト、   │
│  従業員のスキル不足、データのプライバシーとセキュリティ、倫理的な問題など、製造業特有の課題を特定し、その解決   │
│  策を探るための基礎を提供します。                                                                               │
│                                                                                                                 │
│  これらの調査小問を通じて、国内製造業における生成AI導入の現状、利点、課題を包括的に理解することができ、効果的   │
│  な導入戦略の策定に寄与する知見を得ることが期待できます。                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 分析結果を、利点と課題が一目で分かる簡潔な日本語レポートにまとめてください。                             │
│  Agent: 統合担当                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8e08660b-07d8-4de8-a0fd-ad6b7eb6b969                                                                       │
│  Final Output: ## 国内製造業における生成AI導入の利点と課題に関する分析レポート                                  │
│                                                                                                                 │
│  国内製造業における生成AIの導入は、業界や企業規模によって異なる状況にある。ハイテク製造業、特に自動車産業や電   │
│  子機器産業では生成AIの導入が進んでおり、約40%の企業が生成AIを活用している。一方、食品や繊維などの伝統的な製造  │
│  業では導入率が約15%にとどまっている。また、大企業では約60%が生成AIを導入しているのに対し、中小企業では約20%に  │
│  とどまっており、企業規模による差異も顕著である。                                                               │
│                                                                                                                 │
│  生成AIの導入による利点としては、以下の4点が挙げられる。                                                        │
│                                                                                                                 │
│  1.                                                                                                             │
│  **生産性の向上**：生成AIを導入した企業の約70%が生産性の向上を報告しており、平均で約25%の生産性向上が達成され   │
│  ている。生産ラインの最適化や自動化が主な要因である。                                                           │
│  2.                                                                                                             │
│  **コスト削減**：約50%の企業がコスト削減効果を実感しており、平均で約15%のコスト削減が実現している。生成AIによ   │
│  る効率化や無駄の削減が寄与している。                                                                           │
│  3.                                                                                                             │
│  **新製品開発の促進**：約30%の企業が新製品の開発期間を短縮したと報告している。生成AIによるシミュレーションや設  │
│  計支援が開発期間の短縮に貢献している。                                                                         │
│  4.                                                                                                             │
│  **品質管理の強化**：約60%の企業が品質管理の強化に成功しており、生成AIによる異常検知や予測保全が品質向上に寄与  │
│  している。                                                                                                     │
│                                                                                                                 │
│  一方、生成AI導入の課題や制約としては、以下の5点が明らかになった。                                              │
│                                                                                                                 │
│  1.                                                                                                             │
│  **技術的な制約**：約45%の企業が、生成AIの導入に必要な技術的な専門知識やリソースの不足を課題として挙げている。  │
│  特に、中小企業ではこの傾向が顕著である。                                                                       │
│  2.                                                                                                             │
│  **コスト**：約40%の企業が、生成AI導入に伴う初期投資や運用コストの高さを懸念している。中小企業ではこのコスト負  │
│  担が導入の大きな障壁となっている。                                                                             │
│  3.                                                                                                             │
│  **従業員のスキル不足**：約35%の企業が、生成AIを効果的に活用するための従業員のスキル不足を課題として認識してい  │
│  る。データサイエンティストやAIエンジニアの不足が深刻である。                                                   │
│  4.                                                                                                             │
│  **データのプライバシーとセキュリティ**：約30%の企業が、生成AIの導入に伴うデータのプライバシーやセキュリティの  │
│  リスクを懸念している。機密データの漏洩や不正アクセスに対する対策が求められている。                             │
│  5.                                                                         

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 国内製造業における生成AI導入の利点と課題に関する分析レポート

国内製造業における生成AIの導入は、業界や企業規模によって異なる状況にある。ハイテク製造業、特に自動車産業や電子機器産業では生成AIの導入が進んでおり、約40%の企業が生成AIを活用している。一方、食品や繊維などの伝統的な製造業では導入率が約15%にとどまっている。また、大企業では約60%が生成AIを導入しているのに対し、中小企業では約20%にとどまっており、企業規模による差異も顕著である。

生成AIの導入による利点としては、以下の4点が挙げられる。

1. **生産性の向上**：生成AIを導入した企業の約70%が生産性の向上を報告しており、平均で約25%の生産性向上が達成されている。生産ラインの最適化や自動化が主な要因である。
2. **コスト削減**：約50%の企業がコスト削減効果を実感しており、平均で約15%のコスト削減が実現している。生成AIによる効率化や無駄の削減が寄与している。
3. **新製品開発の促進**：約30%の企業が新製品の開発期間を短縮したと報告している。生成AIによるシミュレーションや設計支援が開発期間の短縮に貢献している。
4. **品質管理の強化**：約60%の企業が品質管理の強化に成功しており、生成AIによる異常検知や予測保全が品質向上に寄与している。

一方、生成AI導入の課題や制約としては、以下の5点が明らかになった。

1. **技術的な制約**：約45%の企業が、生成AIの導入に必要な技術的な専門知識やリソースの不足を課題として挙げている。特に、中小企業ではこの傾向が顕著である。
2. **コスト**：約40%の企業が、生成AI導入に伴う初期投資や運用コストの高さを懸念している。中小企業ではこのコスト負担が導入の大きな障壁となっている。
3. **従業員のスキル不足**：約35%の企業が、生成AIを効果的に活用するための従業員のスキル不足を課題として認識している。データサイエンティストやAIエンジニアの不足が深刻である。
4. **データのプライバシーとセキュリティ**：約30%の企業が、生成AIの導入に伴うデータのプライバシーやセキュリティのリスクを懸念している。機密データの漏洩や不正アクセスに対する対策が求められている。
5. **倫理的な問題**：

Trace(trace_id=tr-e5080e80c8bc35742970b5ee8717f6be)

# トレースの確認

左サイドバーの「Experiments」から `/Shared/crewai-observability-demo` を開くと、ネストされたトレースが確認できます。標準のLLM呼び出しが開始と終了だけなのに対し、マルチエージェントではネストされたスパンと分岐が現れ、ブラックボックスだった実行が状態遷移のマップになります。

見るべきポイントは次のとおりです。

- ルートの `Crew.kickoff` (autologが作る本物のスパン) の下に、各タスク・エージェント・LLM呼び出しがネストされているか
- 手動ラッパーの `crew_run_with_metrics` スパンの属性に総実行時間・出力量・数値突き合わせが乗っているか
- `state_handoff` スパンの `handoff.efficiency_score` が極端に低くないか (低い場合はコンテキストの取りこぼし)
- `routing.loop_detected` が `true` になっていないか (同じエージェントの過剰な再実行)
- 各エージェント・LLM呼び出しのレイテンシーにボトルネックがないか

補足: CrewAIインテグレーションは同期実行のみ対応です。`kickoff_async()` ではトレースが取得できないため、検証時は同期の `kickoff()` を使ってください。APIキーは本番ではDatabricksシークレットまたは AI Gateway 経由にし、環境変数へのべた書きは避けてください。

複数サービスにまたがる場合は、MLflowのOpenTelemetry (OTEL) サポートでトレースを Grafana や Datadog などのバックエンドにエクスポートでき、フロントエンドのクリックからエージェントの挙動までを単一のタイムラインで追えます。

# 異常系: メトリクスが点灯する様子を見る

ここまでは正常に完了するパイプラインでした。元記事の主眼は「カメラを設置すること」ではなく「カメラが何を映すか」、つまり subtle な失敗をメトリクスで捕まえることにあります。そこで、今回の逐次パイプラインで意味を持つ2つの失敗モード、すなわちカスケードした数値エラーと、コンテキストの取りこぼし (共有メモリ汚染の前兆) をわざと注入し、対応するメトリクスが点灯する様子を確認します。

LLMの非決定性のため、これらの失敗を狙って毎回再現するのは困難です。そこでここでは失敗を決定論的に注入し、検知ロジックが正しく反応することを示します。各デモは `@mlflow.trace` でスパン化され、正常系と同じエクスペリメントに並んで記録されるので、対比して確認できます。

なお元記事のもう1つの失敗モードであるループ (同じエージェントの過剰な再実行や応答待ちでの停止) は、オーケストレーターが委任先を動的に選ぶスーパーバイザー型や、ツールを持つ構成で問題になります。今回の一本道の逐次構成では構造的に発生しにくいため、ここでは扱いません。`routing.loop_detected` のような指標はそうした構成で効いてきます。

In [0]:
# 失敗モード1: カスケードした数値エラー
# 元記事の例: 専門エージェントが制約を取り違え (年次のはずを四半期で計算) 、
# その誤った数値が下流のレポートに伝播する。レポートの数値とソースの数値を
# 突き合わせ、ソースに存在しない数値 (= でっち上げ/取り違え) をフラグする。

def _significant_nums(text: str) -> set[str]:
    # 数値を正規化し、実質的な数値だけを集合で返す。
    # 1桁の数字 (箇条書き番号や見出し番号など) はノイズになりやすいので除外する。
    out: set[str] = set()
    for t in extract_numeric_tokens(text):
        norm = t.replace(",", "").rstrip(".")
        digits = norm.replace(".", "")
        if len(digits) >= 2:
            out.add(norm)
    return out


@mlflow.trace(
    name="numeric_report_validation",
    span_type="TOOL",
    attributes={"service": "heuristic numeric cross-check (report vs sources)"},
)
def validate_report_numbers_against_sources(final_report: str, *source_texts: str) -> dict:
    report_nums = _significant_nums(final_report)
    source_nums: set[str] = set()
    for s in source_texts:
        source_nums |= _significant_nums(s)
    ungrounded = sorted(report_nums - source_nums)
    span = mlflow.get_current_active_span()
    if span:
        span.set_attributes({
            "validation.report_number_count": len(report_nums),
            "validation.source_number_count": len(source_nums),
            "validation.ungrounded_number_count": len(ungrounded),
            "validation.ungrounded_numbers": ", ".join(ungrounded) if ungrounded else "(none)",
            "validation.passed": len(ungrounded) == 0,
        })
    return {"ungrounded": ungrounded, "passed": len(ungrounded) == 0}


# このセルは「正常 -> 汚染を注入 -> 再検証」の順に進み、
# 数値クロスチェックがカスケードした誤りを捕まえる様子を1セルで再現します。

print("=" * 60)
print("失敗モード1: カスケードした数値エラーの検知デモ")
print("=" * 60)

# ステップ1: 正常系ランの本物の結果 (synthesis = レポート, analysis = ソース) を取り出す
src_analysis = crew_state.get_state("analysis") or ""
src_synthesis = crew_state.get_state("synthesis") or ""
print(f"\n[1] 正常系ランの結果を取得  (ソース {len(src_analysis)}字 / レポート {len(src_synthesis)}字)")

# ステップ2: 正常系レポートを検証してベースラインを取る
clean = validate_report_numbers_against_sources(src_synthesis, src_analysis)
print(f"[2] 正常系レポートを検証     -> passed={clean['passed']} / "
      f"ソースに無い数値={clean['ungrounded'] or 'なし'}")
print("    レポート中の数値はすべてソースに裏づけられており、健全な状態です。")

# ステップ3: ソースに存在しない数値 (4242) を含む一文を注入する
FABRICATED = "なお試算では四半期売上は4242億円に達する見込みです。"
corrupted_report = src_synthesis + FABRICATED
print(f"\n[3] カスケードエラーを注入   -> 追記した一文:「{FABRICATED}」")
print("    アナリスト段が制約を取り違え、ソースに無い数値が紛れ込んだ状況を再現します。")

# ステップ4: 汚染レポートを同じ検証にかける
broken = validate_report_numbers_against_sources(corrupted_report, src_analysis)
print(f"[4] 汚染レポートを検証       -> passed={broken['passed']} / "
      f"ソースに無い数値={broken['ungrounded']}")

# ステップ5: 正常系のベースラインを差し引き、注入で新たに増えた数値だけを取り出す
newly_flagged = sorted(set(broken["ungrounded"]) - set(clean["ungrounded"]))
print(f"\n[5] 正常系のノイズを差し引いて、注入で新たに増えた数値だけを抽出")
print(f"    -> 検知された注入数値: {newly_flagged}")
print("\n結論: でっち上げられた数値 4242 だけが、ソース未裏づけとしてフラグされました。")

失敗モード1: カスケードした数値エラーの検知デモ

[1] 正常系ランの結果を取得  (ソース 1644字 / レポート 1214字)
[2] 正常系レポートを検証     -> passed=True / ソースに無い数値=なし
    レポート中の数値はすべてソースに裏づけられており、健全な状態です。

[3] カスケードエラーを注入   -> 追記した一文:「なお試算では四半期売上は4242億円に達する見込みです。」
    アナリスト段が制約を取り違え、ソースに無い数値が紛れ込んだ状況を再現します。
[4] 汚染レポートを検証       -> passed=False / ソースに無い数値=['4242']

[5] 正常系のノイズを差し引いて、注入で新たに増えた数値だけを抽出
    -> 検知された注入数値: ['4242']

結論: でっち上げられた数値 4242 だけが、ソース未裏づけとしてフラグされました。


[Trace(trace_id=tr-7123385d0186a8c6ddaaa4f51e2ff4b1), Trace(trace_id=tr-c23ada4e07115ebd99bbe66f690e7cfe)]

In [0]:
# 失敗モード2: コンテキストの取りこぼし (ハンドオフ効率の低下)
# 共有メモリ汚染や情報落ちの前兆。前段の出力が次段にどれだけ引き継がれたかを
# トークン重複で見る。次段が受け取るコンテキストを切り詰めて、スコアが落ちるのを確認する。

@mlflow.trace(name="anomaly_handoff_collapse", span_type="CHAIN")
def demo_handoff_collapse() -> dict:
    plan = crew_state.get_state("plan") or ""
    analysis = crew_state.get_state("analysis") or ""

    healthy = round(handoff_efficiency_score(plan, analysis), 3)
    # 次段が先頭150文字しか受け取れなかった状況をシミュレート
    starved = round(handoff_efficiency_score(plan, analysis[:150]), 3)

    span = mlflow.get_current_active_span()
    if span:
        span.set_attributes({
            "handoff.healthy_efficiency_score": healthy,
            "handoff.starved_efficiency_score": starved,
            "handoff.context_truncated": True,
        })
    return {"healthy": healthy, "starved": starved}


scores = demo_handoff_collapse()
print(f"正常なハンドオフ効率: {scores['healthy']}")
print(f"コンテキスト切り詰め後: {scores['starved']}")
print("=> スコアの低下が情報落ちのシグナルになります。")

正常なハンドオフ効率: 0.618
コンテキスト切り詰め後: 0.162
=> スコアの低下が情報落ちのシグナルになります。


Trace(trace_id=tr-29b9a1a814bb8e3d90cd9e75cfd4b471)

# まとめと記事化のヒント

正常系のランと並べると、メトリクスが2つの失敗モードに反応することが確認できます。`numeric_report_validation` スパンの `validation.ungrounded_numbers` にカスケードした数値が現れ、`anomaly_handoff_collapse` でハンドオフ効率が落ちます。

土台のトレース (autologによるネストされたエージェントグラフ) と、その上のメトリクスが揃って初めて、元記事の主旨である「状態遷移を可視化したうえで、挙動の良し悪しを判定する」可観測性が成立します。記事では、正常系と異常系のスパン属性を並べたスクリーンショットを対比で見せると、「カメラを置くだけでなく、それが何を映すか」という流れにそのまま乗れます。

補足として、数値の突き合わせは簡易チェックです。箇条書き番号などの1桁数字は除外し、桁区切りも正規化していますが、言い換えや表記揺れで実害のない数値が残ることはあります。また、今回はソース自体がLLM生成のため、保証しているのは段間の一貫性であって事実性ではありません。本番では、実データを返すツールと突き合わせるとグラウンディング検証に格上げできます。

元記事はPart 1で、ここまでは受動的な観察です。続くPart 2では、安全性の強制・コスト管理・問題の未然防止といった能動的なステアリング (ガバナンス) が扱われます。本ノートブックはその土台に当たります。